# 02 Logistic regression baseline on F0

Step 6. Training runs from `python -m src.models.logreg` (logic in `src/models/logreg.py` and `src/experiment.py`), which appends every fold and the validation result to `results/log.csv`. This notebook only reads the log, so re-running it never retrains.

Inputs are `LinearPipeline` output (provisional V-column list), refit on each fold's training part. C is tuned over {0.01, 0.1, 1, 10} with expanding-window time CV; the best C by mean CV PR-AUC is refit on the full train block and scored on validation.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.results import LOG_PATH
from src.data import ROOT

log = pd.read_csv(LOG_PATH)
lr = log[(log.model == 'logreg') & (log.feature_set == 'F0')].copy()
params = lr['params'].map(json.loads)
lr['C'] = params.map(lambda p: p['C'])
lr['fold'] = params.map(lambda p: p.get('fold'))
assert (log['split'] != 'test').all(), 'test block must stay unscored'
cv, val = lr[lr.split == 'cv'].astype({'fold': int}), lr[lr.split == 'val']
print(len(cv), 'cv rows,', len(val), 'validation row(s)')

## Time-CV PR-AUC per C and fold

In [ ]:
pd.set_option('display.precision', 4)
cv.pivot_table(index='C', columns='fold', values='pr_auc')

In [ ]:
summary = cv.groupby('C').agg(pr_auc_mean=('pr_auc', 'mean'), pr_auc_std=('pr_auc', 'std'),
                              auroc_mean=('auroc', 'mean'), recall_top1pct_mean=('recall_top1pct', 'mean'),
                              runtime_s_mean=('runtime_s', 'mean'))
summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for fold, g in cv.groupby('fold'):
    ax.plot(g['C'], g['pr_auc'], marker='o', alpha=0.4, label=f'fold {fold}')
ax.errorbar(summary.index, summary['pr_auc_mean'], yerr=summary['pr_auc_std'],
            color='black', marker='s', capsize=3, label='mean ± std')
best_c = val['C'].iloc[0]
ax.axvline(best_c, color='grey', ls='--', lw=1, label=f'chosen C = {best_c:g}')
ax.axhline(0.035, color='red', ls=':', lw=1, label='no-skill (0.035)')
ax.set_xscale('log'); ax.set_xlabel('C'); ax.set_ylabel('PR-AUC (time CV)')
ax.set_title('Logistic regression on F0: time-CV PR-AUC by C')
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(ROOT / 'reports' / 'figures' / 'baseline_logreg_cv.png', dpi=150)

## Validation block (best C refit on the full train block)

In [ ]:
val[['date', 'C', 'params', 'pr_auc', 'auroc', 'precision', 'recall', 'recall_top1pct', 'runtime_s']]